## Tool definition

In [3]:
%run langchain_common.py

In [4]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [5]:
@tool("square_root")
def tool1(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [6]:
@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

In [7]:
tool1.invoke({"x": 467})

21.61018278497431

In [8]:
square_root.invoke({"x": 467})

21.61018278497431

## Adding to agents

In [9]:
agent = create_agent(llm_noreason, tools=[tool1],
    system_prompt="You are an arithmetic wizard. Use your tools to calculate the square root and square of any number."
)

In [10]:
question = HumanMessage(content="What is the square root of 467?")

response = agent.invoke(
    {"messages": [question]}
)

pp(response['messages'][-1].content)

'The square root of 467 is approximately 21.61018278497431.'


In [11]:
pp(response['messages'])

[
    HumanMessage(content='What is the square root of 467?', additional_kwargs={}, response_metadata={}, id='4cd5c104-9e7c-4c84-bf02-c2c916eaddd2'),
    AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 308, 'total_tokens': 336, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen35-122b-a10b', 'system_fingerprint': None, 'id': 'chatcmpl_e6690f20-b7e1-4d59-93dc-99fa4400d75a', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ef2ad-e2fa-7c92-93af-4fd9a7461223-0', tool_calls=[{'name': 'square_root', 'args': {'x': '467'}, 'id': 'call_d4a38c51-4021-4b23-95b2-19afcab62678', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 308, 'output_tokens': 28, 'total_tokens': 336, 'input_token_details': {}, 'output_token_details': {}}),
    ToolMessage(content='21.61018278497431', name='square_root', id='f078aadf-

In [15]:
#Retrieve the tool calls from the LLM response
response["messages"][-1].tool_calls

[]

In [16]:
response["messages"][2].tool_calls

AttributeError: 'ToolMessage' object has no attribute 'tool_calls'

## create_agent (under the hood)

In [17]:
# Equivalent to create_agent, but using the bind_tools method on the LLM

llm_with_tools = llm.bind_tools(tools=[tool1])
query = "What is the square root of 900?"
response = llm_with_tools.invoke(query)
pp(response.tool_calls)

[
    {
        'args': {'x': '900'},
        'id': 'call_2d773ed7-3f4e-4ee6-816c-ea0857a979fa',
        'name': 'square_root',
        'type': 'tool_call',
    },
]


In [18]:
# Simple, explicit demo of a tool call lifecycle

query = "What is the square root of 467?"
ai_msg = llm_with_tools.invoke(query)

print("LLM tool_calls payload:")
pp(ai_msg.tool_calls)


LLM tool_calls payload:
[
    {
        'args': {'x': '467'},
        'id': 'call_1a72695c-9563-4215-88de-272324f35d22',
        'name': 'square_root',
        'type': 'tool_call',
    },
]


In [19]:

# Take the first tool call the model requested
tool_call = ai_msg.tool_calls[0]
tool_name = tool_call["name"].lower()
tool_args = tool_call["args"]  # usually a dict, e.g. {"x": 467}

print("\nParsed tool call:")
print("tool_name:", tool_name)
print("tool_args:", tool_args)



Parsed tool call:
tool_name: square_root
tool_args: {'x': '467'}


In [20]:

# Route by tool name, then invoke the Python method with model-provided args
# In Python, this will hold a reference to the selected tool/callable, not a C-style function pointer
tool_registry = {"square_root": square_root}
selected_tool = tool_registry[tool_name]

tool_output = selected_tool.invoke(tool_args)

print("\nPython tool execution result:")
print(tool_output)


Python tool execution result:
21.61018278497431


## agent.invoke (under the hood)

In [21]:
query = "What is the square root of 1000?"
ai_msg = llm_with_tools.invoke(query)

pp(ai_msg)

for tool_call in ai_msg.tool_calls:
    # Look up the actual function from our 'tools' list
    # (In a real app, you'd use a dictionary mapping names to functions)
    selected_tool = {"square_root": square_root}[tool_call["name"].lower()]
    
    # 3. Execute the function with the arguments provided by the LLM
    tool_output = selected_tool.invoke(tool_call["args"])
    
    # 4. Create a ToolMessage to send the result back to the LLM
    # This must include the tool_call_id so the LLM knows which request this satisfies
    tool_message = ToolMessage(
        content=str(tool_output), 
        tool_call_id=tool_call["id"]
    )

# 5. Feed the result back to the LLM so it can give a final natural language answer
final_response = llm_with_tools.invoke([query, ai_msg, tool_message])
pp(final_response.content)

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 286, 'total_tokens': 359, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen35-122b-a10b', 'system_fingerprint': None, 'id': 'chatcmpl_98ddeea4-a90b-46e6-9380-00519fab9571', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ef2ba-c0f9-7df1-aef3-9883165b22a2-0', tool_calls=[{'name': 'square_root', 'args': {'x': '1000'}, 'id': 'call_a8eb7759-759e-4836-bc51-a594564aa03c', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 286, 'output_tokens': 73, 'total_tokens': 359, 'input_token_details': {}, 'output_token_details': {}})
[
    {
        'summary': [
            {
                'text': 'The user asked for the square root of 1000. I called the square_root function with x=1000 and received the result 31.622776601683793. I should provide this answer t